# Financial Sentiment Baseline Model

This notebook loads the local finance sentiment dataset from `Finance/data.csv`, cleans sentence text, vectorizes it with TF-IDF, trains a Logistic Regression baseline, and evaluates sentiment predictions.


In [ ]:
from pathlib import Path
import re

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [ ]:
search_paths = [
    Path.cwd() / "Finance" / "data.csv",
    Path.cwd() / "data.csv",
    Path.cwd().parent / "Finance" / "data.csv",
]

csv_path = next((path.resolve() for path in search_paths if path.exists()), None)

if csv_path is None:
    searched = "\n - ".join(str(path.resolve()) for path in search_paths)
    raise FileNotFoundError(
        "Could not locate Finance/data.csv. Checked:\n - " + searched
    )

df = pd.read_csv(csv_path)
print(f"Using finance dataset: {csv_path}")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


In [ ]:
EXPECTED_COLUMNS = {"Sentence", "Sentiment"}
missing_columns = EXPECTED_COLUMNS.difference(df.columns)

if missing_columns:
    raise ValueError(f"Finance dataset is missing expected columns: {sorted(missing_columns)}")


def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


model_df = df[["Sentence", "Sentiment"]].dropna().copy()
model_df["text"] = model_df["Sentence"].map(clean_text)
model_df["label"] = model_df["Sentiment"].astype(str).str.strip().str.lower()
model_df = model_df[model_df["text"] != ""].copy()
model_df = model_df[["text", "label"]]

print("Label distribution:")
print(model_df["label"].value_counts(dropna=False))
model_df.head()


In [ ]:
X = model_df["text"]
y = model_df["label"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print("Label mapping:", dict(enumerate(label_encoder.classes_)))

label_series = pd.Series(y_encoded)
stratify_labels = label_series if label_series.nunique() > 1 and label_series.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=stratify_labels,
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")


In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=10000, ngram_range=(1, 2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Example finance sentiment predictions
sample_texts = [
    "The company reported stronger-than-expected quarterly earnings and raised guidance.",
    "Shares fell after management warned about weaker demand next quarter.",
    "The board kept its full-year outlook unchanged.",
    "Revenue grew modestly while operating costs remained stable.",
    "The firm is facing heavy losses and may need restructuring.",
]
sample_clean = [clean_text(text) for text in sample_texts]
sample_features = vectorizer.transform(sample_clean)
sample_preds = label_encoder.inverse_transform(model.predict(sample_features))
for text, pred in zip(sample_texts, sample_preds):
    print(f"Text: {text}\nPredicted label: {pred}\n")
